# Mini Project 1 — Analysis Notebook

**Michelle Tran**  
**Dataset: Patient Satisfaction**  
**Date:5/10/26**  

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [ ]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px


print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** *Patient Satisfaction Dataset https://www.kaggle.com/datasets/vdimitrievska/patient-satisfaction-dataset?resource=download*

**Why this dataset:** *I want to explore the factors that affect patient satisfaction in the health care services. This topic ties back to creating a more functional system that considers the diversity of patient needs.*

**Three analytical questions:**

1. *Which services are most strongly associated with overall satisfaction?*
2. *Do personable interactions with healthcare workers, such as doctor communication and staff friendliness, show a stronger association with overall patient satisfaction than operational factors, such as wait times, administrative procedures, and facility quality?*
3. *Do patients who rate clinical quality highly also tend to rate operational factors highly?*

**What a practitioner would do with these findings:** *These findings would help a practitioner understand patient priorities and what is associated with their dissatisfaction in healthcare services.*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [ ]:
# Shows a list of the labeled columns in the pandas dataframe
list(df.columns)

['satisfaction in RM',
 'Check up appointment',
 'Time waiting',
 'Admin procedures',
 'Hygiene and cleaning',
 'Time of appointment',
 'Quality/experience dr.',
 'Specialists avaliable',
 'Communication with dr',
 'Exact diagnosis',
 'Modern equipment',
 'friendly health care workers',
 'lab services',
 'avaliablity of drugs',
 'waiting rooms',
 'hospital rooms quality',
 'parking, playing rooms, caffes']

In [31]:
df.isnull().sum()

satisfaction in RM                1
Check up appointment              1
Time waiting                      1
Admin procedures                  1
Hygiene and cleaning              1
Time of appointment               1
Quality/experience dr.            1
Specialists avaliable             1
Communication with dr             1
Exact diagnosis                   1
Modern equipment                  1
friendly health care workers      1
lab services                      1
avaliablity of drugs              1
waiting rooms                     1
hospital rooms quality            1
parking, playing rooms, caffes    1
dtype: int64

In [ ]:
df = pd.read_csv('patient_satisfaction_dataset.csv')

#The original ratings (1:satisfied, 2: no satisfied, 3:neutral, 4: very satisfied, 5:very unsatisfied) were reordered to (1: very unsatisfied, 2: not satisfied, 3:neutral, 4: satisfied, 5:very satisfied) to make it easier to analyze the data.
RECODE_TO_ORDERED = { # <-- Recoding dictionary
    5: 1,  # very unsatisfied
    2: 2,  # not satisfied
    3: 3,  # neutral
    1: 4,  # satisfied
    4: 5,  # very satisfied
}

for col in df.columns:
    df[col] = df[col].map(RECODE_TO_ORDERED) # .map is used to apply the recoding dictionary I created to each element in the column

# Based on the null results from df.isnull().sum(), I used the dropna() function to remove all rows with missing values.
# Original rows: 453
df = df.dropna()

print(df.shape)
df.head(500)

(452, 17)


,satisfaction in RM,Check up appointment,Time waiting,Admin procedures,Hygiene and cleaning,Time of appointment,Quality/experience dr.,Specialists avaliable,Communication with dr,Exact diagnosis,Modern equipment,friendly health care workers,lab services,avaliablity of drugs,waiting rooms,hospital rooms quality,"parking, playing rooms, caffes"
0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
1,2.0,2.0,4.0,2.0,4.0,2.0,2.0,4.0,2.0,4.0,2.0,4.0,4.0,4.0,4.0,2.0,4.0
2,2.0,4.0,4.0,2.0,2.0,2.0,5.0,5.0,5.0,5.0,4.0,5.0,5.0,4.0,2.0,2.0,2.0
3,2.0,2.0,5.0,4.0,4.0,2.0,5.0,4.0,4.0,5.0,4.0,4.0,5.0,1.0,5.0,4.0,4.0
4,3.0,5.0,4.0,4.0,2.0,4.0,1.0,1.0,1.0,1.0,2.0,1.0,1.0,1.0,4.0,4.0,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
448,2.0,3.0,3.0,3.0,3.0,2.0,2.0,3.0,3.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0,2.0
449,2.0,5.0,2.0,4.0,3.0,4.0,5.0,5.0,3.0,2.0,3.0,2.0,3.0,5.0,2.0,3.0,5.0
450,2.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0,4.0
451,3.0,4.0,3.0,3.0,3.0,3.0,3.0,2.0,4.0,3.0,2.0,3.0,3.0,3.0,3.0,3.0,3.0


In [52]:
# Check column types and missing values
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 452 entries, 0 to 452
Data columns (total 17 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   satisfaction in RM              452 non-null    float64
 1   Check up appointment            452 non-null    float64
 2   Time waiting                    452 non-null    float64
 3   Admin procedures                452 non-null    float64
 4   Hygiene and cleaning            452 non-null    float64
 5   Time of appointment             452 non-null    float64
 6   Quality/experience dr.          452 non-null    float64
 7   Specialists avaliable           452 non-null    float64
 8   Communication with dr           452 non-null    float64
 9   Exact diagnosis                 452 non-null    float64
 10  Modern equipment                452 non-null    float64
 11  friendly health care workers    452 non-null    float64
 12  lab services                    452 non-n

In [53]:
# Summary statistics for numeric columns
df.describe()

,satisfaction in RM,Check up appointment,Time waiting,Admin procedures,Hygiene and cleaning,Time of appointment,Quality/experience dr.,Specialists avaliable,Communication with dr,Exact diagnosis,Modern equipment,friendly health care workers,lab services,avaliablity of drugs,waiting rooms,hospital rooms quality,"parking, playing rooms, caffes"
count,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000,452.000000
mean,2.482301,3.353982,3.351770,3.234513,2.940265,3.278761,3.353982,3.190265,3.110619,3.398230,3.066372,3.117257,3.331858,3.212389,3.050885,3.006637,3.026549
std,0.530361,1.309494,1.281861,1.283653,1.220108,1.310774,1.460009,1.333216,1.400403,1.405026,1.229721,1.363758,1.387481,1.341484,1.213221,1.201422,1.127839
min,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
50%,2.000000,4.000000,4.000000,3.000000,3.000000,4.000000,4.000000,3.000000,3.000000,4.000000,3.000000,3.000000,4.000000,3.000000,3.000000,3.000000,3.000000
75%,3.000000,4.000000,4.000000,4.000000,4.000000,4.000000,5.000000,4.000000,4.000000,5.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
max,4.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


**Your data profile notes:**  
*(Replace this with your observations — what's in the data, what you noticed, what questions it raises.)*

- **Recode:** All columns used the same five-point wording scale (satisfied / not satisfied / neutral / very satisfied / very unsatisfied). Numeric codes were **not** in order of positivity, so the notebook maps them to **1 (worst) through 5 (best)** before summaries and charts. Downstream cells use this recoded `df`.

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [ ]:
sat = "satisfaction in RM"
predictors = [c for c in df.columns if c != sat]

mean_by_factor = df[predictors].mean().sort_values(ascending=False)
plot_df = mean_by_factor.reset_index()
plot_df.columns = ["Factor", "mean_rating"]

fig = px.bar(
    plot_df,
    x="mean_rating",
    y="Factor",
    orientation="h",
    title="Average rating by hospital aspect (which factors score highest vs lowest)",
    labels={
        "mean_rating": "Mean rating (higher = more satisfied with this aspect)",
        "Factor": "Factor",
    },
)
fig.update_yaxes(autorange="reversed")
fig.show()

In [ ]:
# Grouped bars: for Personable and for Operational, two side-by-side bars:
# (1) mean of that group's items across patients, (2) mean satisfaction in RM on the same rows used for that side.
# Correlation (Spearman) between each patient's composite and RM is shown in the title — same scale as ρ, not mixed into the bar heights.

import plotly.express as px
from pathlib import Path

sat = "satisfaction in RM"
personable = [
    "Quality/experience dr.",
    "Admin procedures",
    "Specialists avaliable",
    "friendly health care workers",
    "Communication with dr",
]
operational = [c for c in df.columns if c not in personable and c != sat]

p_rows = df[personable + [sat]].dropna()
o_rows = df[operational + [sat]].dropna()

# One composite score per patient (same rows as the bars), then correlation with RM
comp_p = p_rows[personable].mean(axis=1)
comp_o = o_rows[operational].mean(axis=1)
rho_p = comp_p.corr(p_rows[sat], method="spearman")
rho_o = comp_o.corr(o_rows[sat], method="spearman")

plot_df = pd.DataFrame(
    [
        {
            "Group": "Personable",
            "Metric": "Mean of group items",
            "Value": comp_p.mean(),
        },
        {
            "Group": "Personable",
            "Metric": "Mean satisfaction in RM",
            "Value": p_rows[sat].mean(),
        },
        {
            "Group": "Operational",
            "Metric": "Mean of group items",
            "Value": comp_o.mean(),
        },
        {
            "Group": "Operational",
            "Metric": "Mean satisfaction in RM",
            "Value": o_rows[sat].mean(),
        },
    ]
)

fig = px.bar(
    plot_df,
    x="Group",
    y="Value",
    color="Metric",
    barmode="group",
    title=(
        "What This Chart Shows — Personable Versus Operational: bar height is the sample mean Likert rating (same 1–5 scale) "
        "for each block of survey items side by side with mean overall satisfaction in the room.<br>"
        f"<sup>Strength of association (Spearman ρ, patient composite versus {sat}): personable = {rho_p:.3f}; operational = {rho_o:.3f}</sup>"
    ),
    labels={
        "Value": "",
        "Group": "",
        "Metric": "",
    },
)
fig.update_layout(
    title_x=0.5,
    legend_title_text="Bar type (composite vs RM satisfaction)",
    xaxis_title="Category — personable (clinical/staff-contact items) versus operational (logistics/facility items)",
    yaxis_title="Mean score — units: Likert-type scale 1 (low satisfaction) … 5 (high satisfaction)",
)

_png = Path.cwd() / "personable vs operational.png"
fig.write_image(str(_png), width=1040, height=600, scale=2)

fig.show()

print("Saved chart for repo:", _png.resolve())

print(plot_df)
print("Spearman ρ personable composite vs RM:", round(rho_p, 4))
print("Spearman ρ operational composite vs RM:", round(rho_o, 4))

         Group                   Metric     Value
0   Personable      Mean of group items  3.201327
1   Personable  Mean satisfaction in RM  2.482301
2  Operational      Mean of group items  3.183427
3  Operational  Mean satisfaction in RM  2.482301
Spearman ρ personable composite vs RM: 0.2207
Spearman ρ operational composite vs RM: 0.2022


In [74]:
# Each dot = one patient: composite score vs satisfaction in RM
# (composite = mean of that group's items for that patient, same columns as the two-bar chart above)

import plotly.express as px

sat = "satisfaction in RM"
personable = [
    "Quality/experience dr.",
    "Admin procedures",
    "Specialists avaliable",
    "friendly health care workers",
    "Communication with dr",
]
operational = [c for c in df.columns if c not in personable and c != sat]

pat = df.copy()
pat["personable_avg"] = df[personable].mean(axis=1)
pat["operational_avg"] = df[operational].mean(axis=1)

fig_personable = px.scatter(
    pat,
    x="personable_avg",
    y=sat,
    opacity=0.35,
    title="Each patient: personable composite vs satisfaction in RM",
    labels={
        "personable_avg": "Mean personable rating (this patient)",
        sat: "Satisfaction in RM",
    },
)
fig_personable.show()

fig_operational = px.scatter(
    pat,
    x="operational_avg",
    y=sat,
    opacity=0.35,
    title="Each patient: operational composite vs satisfaction in RM",
    labels={
        "operational_avg": "Mean operational rating (this patient)",
        sat: "Satisfaction in RM",
    },
)
fig_operational.show()

In [ ]:
# Scatter: one point per factor — compares mean aspect rating (x) to mean RM satisfaction (y)
# using only rows where both that aspect and RM satisfaction are non-missing.
# color="Factor" adds a legend that keys each point to its aspect name.

sat = "satisfaction in RM"
predictors = [c for c in df.columns if c != sat]

scatter_rows = []
for p in predictors:
    sub = df[[sat, p]].dropna()
    scatter_rows.append(
        {
            "Factor": p,
            "mean_aspect": sub[p].mean(),
            "mean_rm_satisfaction": sub[sat].mean(),
        }
    )
scatter_df = pd.DataFrame(scatter_rows)

fig2 = px.scatter(
    scatter_df,
    x="mean_aspect",
    y="mean_rm_satisfaction",
    color="Factor",
    hover_name="Factor",
    title="Per factor: mean aspect rating vs mean satisfaction in RM (complete pairs)",
    labels={
        "mean_aspect": "Mean rating on this aspect (1–5)",
        "mean_rm_satisfaction": "Mean satisfaction in RM (1–5)",
        "Factor": "Hospital aspect",
    },
)
fig2.update_layout(
    legend=dict(
        title=dict(text="Aspect (legend)"),
        tracegroupgap=0,
        font=dict(size=10),
        yanchor="top",
        y=1,
        x=1.02,
        xanchor="left",
    ),
    margin=dict(r=200),
)
fig2.show()

**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Process & Conclusions

Process
-what i learned
-my process and how it changed the way i go about it in the future
Conclusions
Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.